# Selective Data Cleaning

This notebook lets you **choose exactly which sheets and columns** to include when merging
public and private school Excel files into clean CSVs.

### Workflow
1. **Run Cell 2** – lists all available sheets so you know what exists.
2. **Edit the config in Cell 1** – pick the sheets and columns you want.
3. **Run Cells 3–5** – merge, preview, and save lean CSVs.

## Cell 1 – Configuration

Edit the variables below, then re-run the rest of the notebook.

In [ ]:
import pandas as pd
import os

OUTPUT_DIR = 'organized_education_data'

# ── Public school files ───────────────────────────────────────────────────────
PUBLIC_FILES = [
    ('2018-2019', f'{OUTPUT_DIR}/public_schools_2018-2019.xlsx'),
    ('2019-2020', f'{OUTPUT_DIR}/public_schools_2019-2020.xlsx'),
    ('2020-2021', f'{OUTPUT_DIR}/public_schools_2020-2021.xlsx'),
]

# Sheets to include from each public school file.
# Set to None to include ALL sheets (not recommended – causes column misalignment).
# Run Cell 2 first to see which sheets are available.
PUBLIC_SHEETS_TO_USE = [
    'Primary Schools by Province',
    'Secondary Schools by Province',
]

# Columns to keep after merging.  Set to None to keep ALL columns.
# After running Cell 3 you can inspect the full column list and narrow it down.
PUBLIC_COLUMNS_TO_KEEP = None

# ── Private school files ──────────────────────────────────────────────────────
PRIVATE_FILES = [
    ('2018-2019', f'{OUTPUT_DIR}/private_schools_2018-2019.xlsx'),
    ('2019-2020', f'{OUTPUT_DIR}/private_schools_2019-2020.xlsx'),
    ('2020-2021', f'{OUTPUT_DIR}/private_schools_2020-2021.xlsx'),
]

# Sheets to include from each private school file.
PRIVATE_SHEETS_TO_USE = [
    'Schools, Classes & Staff by Pro',
]

# Columns to keep after merging.  Set to None to keep ALL columns.
PRIVATE_COLUMNS_TO_KEEP = None

# ── Output file names ─────────────────────────────────────────────────────────
PUBLIC_OUTPUT_CSV  = 'public_cleaned_selective.csv'
PRIVATE_OUTPUT_CSV = 'private_cleaned_selective.csv'

print('✅ Configuration loaded.')

## Cell 2 – List Available Sheets

Run this cell to see which sheets exist in every file before editing the config.

In [ ]:
print('═' * 60)
print('PUBLIC SCHOOL FILES')
print('═' * 60)
for year, path in PUBLIC_FILES:
    xl = pd.ExcelFile(path)
    print(f'\n📂 {year}  ({os.path.basename(path)})')
    for i, name in enumerate(xl.sheet_names):
        marker = '  ✔' if (PUBLIC_SHEETS_TO_USE and name in PUBLIC_SHEETS_TO_USE) else '   '
        print(f'{marker} [{i:2d}] {name}')

print()
print('═' * 60)
print('PRIVATE SCHOOL FILES')
print('═' * 60)
for year, path in PRIVATE_FILES:
    xl = pd.ExcelFile(path)
    print(f'\n📂 {year}  ({os.path.basename(path)})')
    for i, name in enumerate(xl.sheet_names):
        marker = '  ✔' if (PRIVATE_SHEETS_TO_USE and name in PRIVATE_SHEETS_TO_USE) else '   '
        print(f'{marker} [{i:2d}] {name}')

## Cell 3 – Load & Merge Selected Sheets

Reads only the sheets you selected, tags each row with its year, and applies the
optional column filter.  Nothing is saved yet.

In [ ]:
def load_and_merge(files, sheets_to_use, columns_to_keep, label):
    """Load selected sheets from a list of Excel files and merge them.

    Parameters
    ----------
    files : list of (year_str, path) tuples
    sheets_to_use : list[str] | None  – sheet names to include; None = all sheets
    columns_to_keep : list[str] | None  – column names to keep; None = all columns
    label : str  – human-readable name used in status messages
    """
    frames = []
    for year, path in files:
        xl = pd.ExcelFile(path)
        available = xl.sheet_names
        selected  = sheets_to_use if sheets_to_use is not None else available

        missing = [s for s in selected if s not in available]
        if missing:
            print(f'  ⚠️  {year}: sheets not found and skipped: {missing}')
        selected = [s for s in selected if s in available]

        for sheet in selected:
            df = pd.read_excel(path, sheet_name=sheet)
            df.insert(0, 'Year', year)
            df.insert(1, 'Sheet', sheet)
            frames.append(df)
            print(f'  ✅ {year} / {sheet!r:40s}  {len(df):5d} rows × {len(df.columns)} cols')

    if not frames:
        print(f'  ❌ No data loaded for {label}.')
        return pd.DataFrame()

    merged = pd.concat(frames, ignore_index=True)
    print(f'\n  Merged total: {len(merged)} rows × {len(merged.columns)} columns')

    if columns_to_keep is not None:
        # Always keep the Year and Sheet tag columns
        extra = ['Year', 'Sheet']
        keep = extra + [c for c in columns_to_keep if c in merged.columns]
        missing_cols = [c for c in columns_to_keep if c not in merged.columns]
        if missing_cols:
            print(f'  ⚠️  Columns not found and skipped: {missing_cols}')
        merged = merged[keep]
        print(f'  After column filter: {len(merged)} rows × {len(merged.columns)} columns')

    return merged


print('Loading public school data…')
public_df = load_and_merge(PUBLIC_FILES, PUBLIC_SHEETS_TO_USE, PUBLIC_COLUMNS_TO_KEEP, 'public')

print()
print('Loading private school data…')
private_df = load_and_merge(PRIVATE_FILES, PRIVATE_SHEETS_TO_USE, PRIVATE_COLUMNS_TO_KEEP, 'private')

## Cell 4 – Preview Data

Inspect the merged DataFrames before saving.  Use this to decide whether
you need to adjust `PUBLIC_COLUMNS_TO_KEEP` / `PRIVATE_COLUMNS_TO_KEEP` in Cell 1.

In [ ]:
def show_preview(df, label):
    if df.empty:
        print(f'{label}: no data to preview.')
        return
    print(f'{'─' * 60}')
    print(f'{label}  –  {len(df)} rows × {len(df.columns)} columns')
    print(f'{'─' * 60}')
    print(f'Columns ({len(df.columns)}):')
    for col in df.columns:
        non_null = df[col].notna().sum()
        print(f'  {col!s:<45s}  {non_null:5d} non-null')
    print()
    display(df.head(5))

show_preview(public_df,  '📗 Public Schools')
print()
show_preview(private_df, '📘 Private Schools')

## Cell 5 – Save Lean CSVs

Saves the filtered DataFrames to CSV.  Output files are placed next to the
existing `public_cleaned.csv` / `private_cleaned.csv` for easy comparison.

In [ ]:
def save_csv(df, filename, label):
    if df.empty:
        print(f'⚠️  {label}: nothing to save.')
        return
    df.to_csv(filename, index=False)
    size_kb = os.path.getsize(filename) / 1024
    print(f'✅ Saved {label} → {filename}')
    print(f'   {len(df)} rows × {len(df.columns)} columns  |  {size_kb:.1f} KB')

save_csv(public_df,  PUBLIC_OUTPUT_CSV,  'Public Schools')
print()
save_csv(private_df, PRIVATE_OUTPUT_CSV, 'Private Schools')